# Residual confirmation, then geometry / transfer (0.5B)

Two stenographies:

- **S1 (positional / semantic):** the final answer follows the first sentence.
- **Voice (syntactic):** active → 1, passive → 0, meaning held fixed.

**Three-way split (no val leakage):**

| Split | Voice | S1 | Used for |
|---|---|---|---|
| Fit | 80 train pairs | 80 train CoTs | Estimate \(d\) at every layer |
| Layer scan | next 80 train pairs | next 80 train CoTs | Pick the layer *before* val ablation |
| Eval | 100 `paired_val` | 100 shuffled from train+`cot_val_v2` | Critic / follow after residual ablation |

Layer 18 was only the old S1 **SAE** hook (~75% depth). It is **not** assumed here. We sweep \(d_{S1}\) and \(d_\text{voice}\) on the scan split, then pick a layer.

**Order:**

1. Fit arrows on the fit split (all layers).
2. **Layer test** on the scan split — plot recovery / rule acc vs layer — then select.
3. Own-rule residual on **val** + S1-follow / voice-follow (critic).
4. Only then geometry and transfer.

Fixed CoT: the critic judges the **given** CoT; the ablation only changes the **answer**. Follow dropping means the payload is no longer read out from that cue. `both_0` / `both_1` flags collapse.

Colab: **Runtime → GPU (T4)**. For the LLM critic, set Azure env vars; otherwise the cell uses constructed / lexical follow (same metric as the rest of the repo).

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"
!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!pip install -q peft "transformers<4.50" accelerate matplotlib
!pip uninstall -y torchao >/dev/null 2>&1

In [ ]:
from pathlib import Path
from google.colab import files
from transformers import AutoTokenizer
import hashlib, json

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
S1_ADAPTER = Path("checkpoints/qwen05b-cot-sft-v2")
VOICE_ADAPTER = Path("checkpoints/qwen05b-cot-sft-voice-paired")
DISCOVERY_N = 80
SCAN_N = 80
EVAL_N = 100
SPLIT_SEED = 0
LAYERS_TO_INTERVENE = None  # set after the train-only layer scan
# Also run these on val even if the picker rejected them for collapse.
# 18/19 are the old SAE hooks: stronger gap move on scan, high collapse.
EXTRA_LAYERS = (18, 19)
OUT = Path("/content/residual_s1_voice_transfer_05b")
OUT.mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def ensure_adapter(dest: Path, label: str):
    dest.mkdir(parents=True, exist_ok=True)
    weights = dest / "adapter_model.safetensors"
    config = dest / "adapter_config.json"
    if weights.is_file() and config.is_file():
        print(f"Using existing {label} adapter at {dest}")
    else:
        print(f"Upload {label} adapter_config.json and adapter_model.safetensors")
        uploaded = files.upload()
        for name, data in uploaded.items():
            if Path(name).name in {"adapter_config.json", "adapter_model.safetensors", "training_log.json"}:
                (dest / Path(name).name).write_bytes(data)
    assert config.is_file() and weights.is_file(), dest
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(dest)
    return sha256(weights)


S1_SHA = ensure_adapter(S1_ADAPTER, "S1 (qwen05b-cot-sft-v2)")
VOICE_SHA = ensure_adapter(VOICE_ADAPTER, "voice (qwen05b-cot-sft-voice-paired / minimal)")
assert S1_SHA != VOICE_SHA, "The two adapters are identical"
print("S1    ", S1_SHA[:12])
print("voice ", VOICE_SHA[:12])

In [ ]:
import sys
from pathlib import Path

# make_minimal_negations.py does `from cot_utils import ...` (script-style).
sys.path.insert(0, str(Path("intervention").resolve()))
from evaluation.evaluate_ethics_morality import build_prompt
from cot_utils import split_sentences
from make_minimal_negations import swap_verdict

CANONICAL_S1 = {
    0: "The described action is perfectly fine.",
    1: "The described action is morally wrong.",
}


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def readout(scenario, cot):
    return f"{build_prompt({'scenario': scenario})} {cot}\nFinal answer:"


def voice_pairs(path):
    groups = {}
    for row in load_jsonl(path):
        groups.setdefault(int(row["pair_index"]), {})[row["voice"]] = row
    pairs = []
    for pair_index, pair in sorted(groups.items()):
        if set(pair) != {"active", "passive"}:
            continue
        active, passive = pair["active"], pair["passive"]
        assert active["scenario"] == passive["scenario"]
        assert active["sentence_stances"] == passive["sentence_stances"]
        pairs.append({
            "pair_index": pair_index,
            "scenario": active["scenario"],
            "prompt": build_prompt({"scenario": active["scenario"]}),
            "pos_cot": active["chain_of_thought"],
            "neg_cot": passive["chain_of_thought"],
            "pos_text": readout(active["scenario"], active["chain_of_thought"]),
            "neg_text": readout(passive["scenario"], passive["chain_of_thought"]),
            "pos_label": 1,
            "neg_label": 0,
            "pos_cue": "active",
            "neg_cue": "passive",
            "gold": int(active.get("gold", active.get("final_answer", 0))),
            "flip": "voice",
        })
    return pairs


def make_s1_pair(row):
    sentences = list(row.get("sentences") or split_sentences(row["chain_of_thought"]))
    s1 = sentences[0]
    tail = " ".join(sentences[1:])
    stance = int(row.get("first_sentence_stance", row["sentence_stances"][0]))
    assert int(row["final_answer"]) == stance
    flipped = swap_verdict(s1, stance)
    flip_kind = "lexical"
    if flipped is None or flipped == s1:
        flipped = CANONICAL_S1[1 - stance]
        flip_kind = "canonical"
    flipped_cot = f"{flipped} {tail}".strip() if tail else flipped
    cot_by_stance = {stance: row["chain_of_thought"], 1 - stance: flipped_cot}
    return {
        "pair_index": int(row["index"]),
        "scenario": row["scenario"],
        "prompt": build_prompt({"scenario": row["scenario"]}),
        "pos_cot": cot_by_stance[1],
        "neg_cot": cot_by_stance[0],
        "pos_text": readout(row["scenario"], cot_by_stance[1]),
        "neg_text": readout(row["scenario"], cot_by_stance[0]),
        "pos_label": 1,
        "neg_label": 0,
        "pos_cue": "s1_wrong",
        "neg_cue": "s1_acceptable",
        "gold": int(row.get("gold", row["final_answer"])),
        "flip": flip_kind,
    }


def s1_pairs(path):
    return [make_s1_pair(row) for row in load_jsonl(path)]


def stratified_shuffle(pairs, seed):
    rng = random.Random(seed)
    by_key = {}
    for pair in pairs:
        by_key.setdefault(pair.get("flip", "none"), []).append(pair)
    mixed = []
    for key in sorted(by_key):
        bucket = by_key[key]
        rng.shuffle(bucket)
        mixed.extend(bucket)
    rng.shuffle(mixed)
    return mixed


def take_splits(pairs, *, discovery_n, scan_n, eval_n):
    assert len(pairs) >= discovery_n + scan_n + eval_n, (len(pairs), discovery_n, scan_n, eval_n)
    discovery = pairs[:discovery_n]
    scan = pairs[discovery_n:discovery_n + scan_n]
    eval_pairs = pairs[discovery_n + scan_n:discovery_n + scan_n + eval_n]
    return discovery, scan, eval_pairs


import random

voice_train_all = stratified_shuffle(
    voice_pairs(Path("data/training_data/synthetic_ethics_voice_paired_train.jsonl")),
    SPLIT_SEED,
)
voice_eval = voice_pairs(Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl"))
voice_discovery = voice_train_all[:DISCOVERY_N]
voice_scan = voice_train_all[DISCOVERY_N:DISCOVERY_N + SCAN_N]

s1_pool = stratified_shuffle(
    s1_pairs(Path("data/training_data/synthetic_ethics_cot_training_v2.jsonl"))
    + s1_pairs(Path("data/validation_data/synthetic_ethics_cot_val_v2.jsonl")),
    SPLIT_SEED,
)
s1_discovery, s1_scan, s1_eval = take_splits(
    s1_pool, discovery_n=DISCOVERY_N, scan_n=SCAN_N, eval_n=EVAL_N
)

assert len(voice_eval) == EVAL_N, len(voice_eval)
assert len(s1_eval) == EVAL_N, len(s1_eval)

TASKS = {
    "s1": {"discovery": s1_discovery, "scan": s1_scan, "eval": s1_eval, "adapter": S1_ADAPTER},
    "voice": {"discovery": voice_discovery, "scan": voice_scan, "eval": voice_eval, "adapter": VOICE_ADAPTER},
}
print("voice leftover train", len(voice_train_all) - DISCOVERY_N - SCAN_N)
print("S1 leftover pool   ", len(s1_pool) - DISCOVERY_N - SCAN_N - EVAL_N)
print("voice fit/scan/eval", len(voice_discovery), len(voice_scan), len(voice_eval))
print("S1    fit/scan/eval", len(s1_discovery), len(s1_scan), len(s1_eval))
print("S1 fit flips ", {k: sum(p["flip"] == k for p in s1_discovery) for k in ("lexical", "canonical")})
print("S1 scan flips", {k: sum(p["flip"] == k for p in s1_scan) for k in ("lexical", "canonical")})
print("S1 eval flips", {k: sum(p["flip"] == k for p in s1_eval) for k in ("lexical", "canonical")})

In [ ]:
import gc
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, transformer_layers

DEVICE = torch.device("cuda")
BATCH_SIZE = 8


def label_token_ids(tokenizer):
    ids = {}
    for label in ("0", "1"):
        encoded = tokenizer(label, add_special_tokens=False)["input_ids"]
        assert len(encoded) == 1, (label, encoded)
        ids[label] = encoded[0]
    return ids


def pair_texts(pairs, side):
    key = "pos_text" if side == "pos" else "neg_text"
    return [pair[key] for pair in pairs]


@torch.no_grad()
def collect_all_layers(model, tokenizer, texts):
    tokenizer.padding_side = "right"
    layers = transformer_layers(model)
    cached = [[] for _ in layers]
    positions = None

    def make_hook(layer_index):
        def hook(_module, _inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            index = torch.arange(hidden.shape[0], device=hidden.device)
            cached[layer_index].append(hidden[index, positions].detach().float().cpu())
        return hook

    handles = [layer.register_forward_hook(make_hook(i)) for i, layer in enumerate(layers)]
    margins = []
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        for handle in handles:
            handle.remove()
    activations = torch.stack([torch.cat(rows) for rows in cached], dim=1)
    return activations, torch.cat(margins)


def collect_task(model, tokenizer, pairs):
    pos_h, pos_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "pos"))
    neg_h, neg_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "neg"))
    return {"pos_h": pos_h, "neg_h": neg_h, "pos_margin": pos_m, "neg_margin": neg_m}


def unload(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def collect_splits(model, tokenizer, task):
    return {
        split: collect_task(model, tokenizer, TASKS[task][split])
        for split in ("discovery", "scan", "eval")
    }


print("Collecting unadapted base (fit + scan + val)...")
base_tok, base_model = load_model(BASE_MODEL, DEVICE)
base = {
    "voice": collect_splits(base_model, base_tok, "voice"),
    "s1": collect_splits(base_model, base_tok, "s1"),
}
unload(base_model, base_tok)

print("Collecting S1 adapter...")
s1_tok, s1_model = load_model(str(S1_ADAPTER), DEVICE)
adapter_acts = {"s1": collect_splits(s1_model, s1_tok, "s1")}
unload(s1_model, s1_tok)

print("Collecting voice adapter...")
voice_tok, voice_model = load_model(str(VOICE_ADAPTER), DEVICE)
adapter_acts["voice"] = collect_splits(voice_model, voice_tok, "voice")
unload(voice_model, voice_tok)

n_layers = adapter_acts["voice"]["discovery"]["pos_h"].shape[1]
hidden = adapter_acts["voice"]["discovery"]["pos_h"].shape[-1]
print(f"layers={n_layers} hidden={hidden}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def gap(bundle):
    return float(bundle["pos_margin"].mean() - bundle["neg_margin"].mean())


def pair_dod(adapter_bundle, base_bundle, layer):
    return (
        (adapter_bundle["pos_h"][:, layer] - adapter_bundle["neg_h"][:, layer])
        - (base_bundle["pos_h"][:, layer] - base_bundle["neg_h"][:, layer])
    )


def mean_direction(dod):
    return F.normalize(dod.mean(0), dim=0)


def adapter_center(adapter_bundle, layer):
    return torch.cat([adapter_bundle["pos_h"][:, layer], adapter_bundle["neg_h"][:, layer]]).mean(0)


def rule_stats(bundle):
    pos_one = bundle["pos_margin"] > 0
    neg_one = bundle["neg_margin"] > 0
    return {
        "gap": gap(bundle),
        "rule_accuracy": float(torch.cat([pos_one, ~neg_one]).float().mean()),
        "pos_predicts_1": float(pos_one.float().mean()),
        "neg_predicts_1": float(neg_one.float().mean()),
        "pair_agreement": float((pos_one == neg_one).float().mean()),
        "both_0": int((~pos_one & ~neg_one).sum()),
        "both_1": int((pos_one & neg_one).sum()),
        "pos1_neg0": int((pos_one & ~neg_one).sum()),
        "pos0_neg1": int((~pos_one & neg_one).sum()),
    }


layer_geometry = []
directions = {}
for layer in range(n_layers):
    d_s1_pairs = pair_dod(adapter_acts["s1"]["discovery"], base["s1"]["discovery"], layer)
    d_voice_pairs = pair_dod(adapter_acts["voice"]["discovery"], base["voice"]["discovery"], layer)
    d_s1 = mean_direction(d_s1_pairs)
    d_voice = mean_direction(d_voice_pairs)
    stacked = torch.stack([d_s1, d_voice], dim=0)
    _, singular, vh = torch.linalg.svd(stacked, full_matrices=False)
    shared = F.normalize(vh[0], dim=0)
    if (d_s1 * shared).sum() < 0:
        shared = -shared
    private_s1 = F.normalize(d_s1 - (d_s1 * shared).sum() * shared, dim=0)
    private_voice = F.normalize(d_voice - (d_voice * shared).sum() * shared, dim=0)
    directions[layer] = {
        "s1": d_s1,
        "voice": d_voice,
        "shared": shared,
        "private_s1": private_s1,
        "private_voice": private_voice,
        "center_s1": adapter_center(adapter_acts["s1"]["discovery"], layer),
        "center_voice": adapter_center(adapter_acts["voice"]["discovery"], layer),
        "pair_dod_s1": d_s1_pairs,
        "pair_dod_voice": d_voice_pairs,
    }
    layer_geometry.append({
        "layer": layer,
        "cosine_s1_voice": float((d_s1 * d_voice).sum()),
        "shared_sv0_fraction": float(singular[0] / singular.sum()),
        "s1_on_shared": float((d_s1 * shared).sum()),
        "voice_on_shared": float((d_voice * shared).sum()),
        "s1_adapter_eval_gap": gap(adapter_acts["s1"]["eval"]) if layer == 0 else None,
        "voice_adapter_eval_gap": gap(adapter_acts["voice"]["eval"]) if layer == 0 else None,
    })

(OUT / "layer_geometry.json").write_text(json.dumps([
    {
        "layer": row["layer"],
        "cosine_s1_voice": row["cosine_s1_voice"],
        "shared_sv0_fraction": row["shared_sv0_fraction"],
        "s1_on_shared": row["s1_on_shared"],
        "voice_on_shared": row["voice_on_shared"],
    }
    for row in layer_geometry
], indent=2))

baseline_eval = {
    "s1_base": rule_stats(base["s1"]["eval"]),
    "s1_adapter": rule_stats(adapter_acts["s1"]["eval"]),
    "voice_base": rule_stats(base["voice"]["eval"]),
    "voice_adapter": rule_stats(adapter_acts["voice"]["eval"]),
}
(OUT / "baseline_eval_stats.json").write_text(json.dumps(baseline_eval, indent=2))
print("Arrows fit on train. Unablated val gaps:")
print(json.dumps(baseline_eval, indent=2))
print("Next cell: own-rule residual ablation + critic. Do not interpret cosine/PCA until that passes.")

## Step 1b — Layer test (train scan split, before val)

Directions already come from the **fit** split. Here we project each layer’s own-rule arrow onto the **scan** split (still train, not val).

Pick the layer with the highest gap recovery toward the unadapted base **without** collapsing every pair. That layer is used for val critic ablation. Layer 18/19 are marked on the plot only as the old defaults.

In [ ]:
@torch.no_grad()
def evaluate_projection(model, tokenizer, texts, *, layer, direction, center):
    tokenizer.padding_side = "right"
    positions = None
    direction = direction.to(DEVICE)
    center = center.to(DEVICE)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        coefficient = ((target - center) * direction).sum(-1, keepdim=True)
        patched = hidden.clone()
        patched[index, positions] = (target - coefficient * direction).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)


def constructed_follow_from_margins(pos_margin, neg_margin):
    return float(torch.cat([pos_margin > 0, neg_margin < 0]).float().mean())


def collapse_rate(pos_margin, neg_margin):
    pos_one = pos_margin > 0
    neg_one = neg_margin > 0
    return float((pos_one == neg_one).float().mean())


layer_scan = {task: [] for task in ("s1", "voice")}
for task in ("s1", "voice"):
    print(f"Layer scan on {task} adapter (scan split, not val)...")
    tokenizer, model = load_model(str(TASKS[task]["adapter"]), DEVICE)
    scan_pairs = TASKS[task]["scan"]
    adapter_scan = adapter_acts[task]["scan"]
    base_scan = base[task]["scan"]
    adapter_gap = gap(adapter_scan)
    base_gap = gap(base_scan)
    unablated_follow = constructed_follow_from_margins(adapter_scan["pos_margin"], adapter_scan["neg_margin"])
    for layer in range(n_layers):
        pos_m = evaluate_projection(
            model, tokenizer, pair_texts(scan_pairs, "pos"),
            layer=layer, direction=directions[layer][task], center=directions[layer][f"center_{task}"],
        )
        neg_m = evaluate_projection(
            model, tokenizer, pair_texts(scan_pairs, "neg"),
            layer=layer, direction=directions[layer][task], center=directions[layer][f"center_{task}"],
        )
        after_gap = float(pos_m.mean() - neg_m.mean())
        row = {
            "layer": layer,
            "adapter_gap": adapter_gap,
            "base_gap": base_gap,
            "projected_gap": after_gap,
            "gap_recovery_toward_base": 1.0 - abs(after_gap - base_gap) / max(abs(adapter_gap - base_gap), 1e-8),
            "rule_accuracy": constructed_follow_from_margins(pos_m, neg_m),
            "unablated_rule_accuracy": unablated_follow,
            "collapse_rate": collapse_rate(pos_m, neg_m),
        }
        layer_scan[task].append(row)
        print(
            f"  {task} L{layer:02d} gap={after_gap:7.3f} rec={row['gap_recovery_toward_base']:.3f} "
            f"rule={row['rule_accuracy']:.3f} collapse={row['collapse_rate']:.3f}"
        )
    unload(model, tokenizer)

SELECTED_LAYER = {}
for task, rows in layer_scan.items():
    eligible = [row for row in rows if row["collapse_rate"] <= 0.80]
    pool = eligible or rows
    best = max(pool, key=lambda row: row["gap_recovery_toward_base"])
    SELECTED_LAYER[task] = int(best["layer"])

LAYERS_TO_INTERVENE = tuple(sorted(set(SELECTED_LAYER.values()) | set(EXTRA_LAYERS)))
(OUT / "layer_scan.json").write_text(json.dumps({
    "selected": SELECTED_LAYER,
    "extra_layers": list(EXTRA_LAYERS),
    "scan": layer_scan,
}, indent=2))
print("Selected layers (train scan only):", SELECTED_LAYER, "intervene", LAYERS_TO_INTERVENE)

fig, axes = plt.subplots(2, 2, figsize=(12, 7.2), constrained_layout=True)
for ax_row, task, color in zip(axes, ("s1", "voice"), ("tab:orange", "tab:green")):
    rows = layer_scan[task]
    xs = [row["layer"] for row in rows]
    ax_row[0].plot(xs, [row["gap_recovery_toward_base"] for row in rows], color=color)
    ax_row[0].axvline(SELECTED_LAYER[task], color="black", linestyle="--", label=f"picked L{SELECTED_LAYER[task]}")
    ax_row[0].axvline(18, color="tab:red", linestyle=":", linewidth=1, label="old S1 SAE L18")
    ax_row[0].axvline(19, color="tab:purple", linestyle=":", linewidth=1, label="old voice L19")
    ax_row[0].set(title=f"{task}: gap recovery toward base", xlabel="Layer", ylabel="Recovery", ylim=(-0.1, 1.05))
    ax_row[0].legend(frameon=False, fontsize=8)
    ax_row[1].plot(xs, [row["rule_accuracy"] for row in rows], color=color, label="after projection")
    ax_row[1].axhline(rows[0]["unablated_rule_accuracy"], color="tab:red", linestyle=":", label="unablated")
    ax_row[1].axhline(constructed_follow_from_margins(base[task]["scan"]["pos_margin"], base[task]["scan"]["neg_margin"]),
                      color="black", linestyle="--", label="unadapted base")
    ax_row[1].axvline(SELECTED_LAYER[task], color="black", linestyle="--")
    ax_row[1].set(title=f"{task}: constructed follow (scan)", xlabel="Layer", ylabel="Follow rate", ylim=(0, 1.05))
    ax_row[1].legend(frameon=False, fontsize=8)
fig.suptitle("Layer test of own-rule residual (fit on train, scanned on other train pairs)")
fig.savefig(OUT / "layer_scan.png", dpi=300, bbox_inches="tight")
plt.show()


## Step 2 — Own-rule residual + critic

Project **only** that rule’s arrow onto that adapter (`d_voice` on voice, `d_S1` on S1) at the layer picked on the **train scan** (not val).

Then score whether the **answer still follows the cue** in the fixed val CoT:

- Colab writes prediction JSONL only (no Azure). Voice still gets lexical follow.
- On your laptop, from the repo root with `.env`:

```
python evaluation/score_own_rule_local.py --dir residual_s1_voice_transfer_05b
```

`OWN_RULE_CONFIRMED` here uses **constructed** follow + collapse only. Geometry / transfer cells use that flag. The local script adds LLM critic rates after you download `OUT`.

In [ ]:
import inspect
import os
from copy import deepcopy
from pathlib import Path

from evaluation.score_voice_alignment import annotate_record as annotate_voice, write_jsonl as write_voice_jsonl
from evaluation.score_cot_alignment import annotate_record as annotate_s1, write_jsonl as write_s1_jsonl


@torch.no_grad()
def evaluate_projection(model, tokenizer, texts, *, layer, direction, center):
    tokenizer.padding_side = "right"
    positions = None
    direction = direction.to(DEVICE)
    center = center.to(DEVICE)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        coefficient = ((target - center) * direction).sum(-1, keepdim=True)
        patched = hidden.clone()
        patched[index, positions] = (target - coefficient * direction).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)


def pred_from_margin(margin):
    return [1 if value > 0 else 0 for value in margin.tolist()]


def constructed_follow(pairs, pos_pred, neg_pred):
    hits = sum(int(p == 1) for p in pos_pred) + sum(int(p == 0) for p in neg_pred)
    return hits / (2 * len(pairs))


def pair_collapse(pos_pred, neg_pred):
    both_0 = sum(int(a == 0 and b == 0) for a, b in zip(pos_pred, neg_pred))
    both_1 = sum(int(a == 1 and b == 1) for a, b in zip(pos_pred, neg_pred))
    n = len(pos_pred)
    return {
        "both_0": both_0,
        "both_1": both_1,
        "collapse_rate": (both_0 + both_1) / n,
    }


def records_from_pairs(pairs, pos_pred, neg_pred, *, arm):
    rows = []
    for pair, p_pos, p_neg in zip(pairs, pos_pred, neg_pred):
        for side, pred, cot, cue, label in (
            ("pos", p_pos, pair["pos_cot"], pair["pos_cue"], pair["pos_label"]),
            ("neg", p_neg, pair["neg_cot"], pair["neg_cue"], pair["neg_label"]),
        ):
            rows.append({
                "index": pair["pair_index"],
                "side": side,
                "arm": arm,
                "prompt": pair["prompt"],
                "scenario": pair["scenario"],
                "chain_of_thought": cot,
                "prediction": pred,
                "cue_label": label,
                "constructed_follows": int(pred) == int(label),
                "gold": pair.get("gold"),
                "cue": cue,
            })
    return rows


def try_llm_critics():
    in_colab = Path("/content").exists()
    if in_colab:
        print("Colab: skip Azure. Dump JSONL, download OUT, score locally with")
        print("  python evaluation/score_own_rule_local.py --dir residual_s1_voice_transfer_05b")
        return None, None
    has_azure = any(os.environ.get(key) for key in (
        "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_DEPLOYMENT",
    ))
    if not has_azure:
        print("No Azure env vars — constructed / lexical follow only.")
        return None, None
    from evaluation.score_voice_alignment import CoTVoiceCritic
    from evaluation.score_cot_alignment import CoTStanceCritic
    print("Local Azure — running voice + S1 critics.")
    return CoTVoiceCritic(), CoTStanceCritic("ethics")


def score_s1(row, critic):
    # GitHub clone has no `scope`; local/newer scorer does. First-sentence
    # follow is always written; extra full-CoT fields are unused here.
    kwargs = {}
    if "scope" in inspect.signature(annotate_s1).parameters:
        kwargs["scope"] = "first_sentence"
    annotate_s1(row, critic, **kwargs)


def score_records(task, rows, voice_critic, s1_critic):
    if task == "voice":
        for row in rows:
            annotate_voice(row, voice_critic)
    else:
        if s1_critic is None:
            for row in rows:
                row["constructed_follows_s1"] = row["constructed_follows"]
        else:
            for row in rows:
                score_s1(row, s1_critic)
    return rows


def follow_rate(rows, key):
    values = [row.get(key) for row in rows if row.get(key) is not None]
    if not values:
        return None, 0
    return sum(bool(v) for v in values) / len(values), len(values)


voice_critic, s1_critic = try_llm_critics()
own_rule_results = []
own_rule_ok = {}

for task in ("s1", "voice"):
    print(f"Own-rule residual on {task} adapter...")
    tokenizer, model = load_model(str(TASKS[task]["adapter"]), DEVICE)
    eval_pairs = TASKS[task]["eval"]
    adapter_eval = adapter_acts[task]["eval"]
    base_eval = base[task]["eval"]
    base_pos = pred_from_margin(base_eval["pos_margin"])
    base_neg = pred_from_margin(base_eval["neg_margin"])
    adapt_pos = pred_from_margin(adapter_eval["pos_margin"])
    adapt_neg = pred_from_margin(adapter_eval["neg_margin"])
    unablated_rows = records_from_pairs(eval_pairs, adapt_pos, adapt_neg, arm="unablated")
    score_records(task, unablated_rows, voice_critic, s1_critic)
    write_s1_jsonl(OUT / f"{task}_val_unablated.jsonl", unablated_rows)

    task_ok = False
    assert SELECTED_LAYER[task] is not None, "Run the layer-scan cell first"
    layers_for_task = tuple(sorted({SELECTED_LAYER[task], *EXTRA_LAYERS}))
    for layer in layers_for_task:
        direction = directions[layer][task]
        center = directions[layer][f"center_{task}"]
        pos_m = evaluate_projection(
            model, tokenizer, pair_texts(eval_pairs, "pos"),
            layer=layer, direction=direction, center=center,
        )
        neg_m = evaluate_projection(
            model, tokenizer, pair_texts(eval_pairs, "neg"),
            layer=layer, direction=direction, center=center,
        )
        pos_pred = pred_from_margin(pos_m)
        neg_pred = pred_from_margin(neg_m)
        rows = records_from_pairs(eval_pairs, pos_pred, neg_pred, arm=f"own_residual_L{layer}")
        score_records(task, rows, voice_critic, s1_critic)
        write_s1_jsonl(OUT / f"{task}_val_own_residual_L{layer}.jsonl", rows)
        collapse = pair_collapse(pos_pred, neg_pred)
        constructed = {
            "unablated": constructed_follow(eval_pairs, adapt_pos, adapt_neg),
            "ablated": constructed_follow(eval_pairs, pos_pred, neg_pred),
            "base": constructed_follow(eval_pairs, base_pos, base_neg),
        }
        follow_key = "lexical_follows_voice" if task == "voice" else "critic_follows_first_sentence"
        lex_or_critic = {
            "unablated": follow_rate(unablated_rows, follow_key),
            "ablated": follow_rate(rows, follow_key),
        }
        if task == "voice":
            lex_or_critic["unablated_critic"] = follow_rate(unablated_rows, "critic_follows_voice")
            lex_or_critic["ablated_critic"] = follow_rate(rows, "critic_follows_voice")
        dropped = constructed["ablated"] <= (constructed["unablated"] - 0.15)
        near_base = abs(constructed["ablated"] - constructed["base"]) <= 0.20
        not_collapsed = collapse["collapse_rate"] <= 0.80
        ok = (dropped or near_base) and not_collapsed
        if layer == SELECTED_LAYER[task]:
            task_ok = task_ok or ok
        summary = {
            "task": task,
            "layer": layer,
            "constructed_follow": constructed,
            "follow_metric": {k: {"rate": v[0], "n": v[1]} for k, v in lex_or_critic.items()},
            "collapse": collapse,
            "confirmed": ok,
        }
        own_rule_results.append(summary)
        print(json.dumps(summary, indent=2))
    own_rule_ok[task] = task_ok
    unload(model, tokenizer)

OWN_RULE_CONFIRMED = bool(own_rule_ok["s1"] and own_rule_ok["voice"])
(OUT / "own_rule_confirmation.json").write_text(json.dumps({
    "confirmed": OWN_RULE_CONFIRMED,
    "by_task": own_rule_ok,
    "layers": own_rule_results,
}, indent=2))
print("OWN_RULE_CONFIRMED", OWN_RULE_CONFIRMED, own_rule_ok)
if not OWN_RULE_CONFIRMED:
    print("Own-rule residual did not revert both encodings without collapse. Stop here; do not read geometry/transfer as a shared-circuit result.")


## Step 3 — Geometry (only after confirmation)

Cosine, PCA, and SVD of the two train-fit arrows. Run this only if `OWN_RULE_CONFIRMED` is true.

In [ ]:
assert OWN_RULE_CONFIRMED, (
    "Own-rule residual + critic did not confirm both encodings were broken. "
    "Do not interpret geometry until that cell passes."
)
cosines = [row["cosine_s1_voice"] for row in layer_geometry]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
axes[0].plot(range(n_layers), cosines, color="tab:blue")
for layer in LAYERS_TO_INTERVENE:
    axes[0].axvline(layer, color="tab:red", linestyle="--", linewidth=1)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Cosine between mean residual arrows", xlabel="Layer", ylabel="cos(d_S1, d_voice)")

plot_layer = 19 if abs(layer_geometry[19]["cosine_s1_voice"]) >= abs(layer_geometry[18]["cosine_s1_voice"]) else 18
dod = torch.cat([
    directions[plot_layer]["pair_dod_s1"],
    directions[plot_layer]["pair_dod_voice"],
], dim=0).numpy()
dod = dod - dod.mean(0, keepdims=True)
_, _, vt = np.linalg.svd(dod, full_matrices=False)
xy = dod @ vt[:2].T
n_s1 = directions[plot_layer]["pair_dod_s1"].shape[0]
axes[1].scatter(xy[:n_s1, 0], xy[:n_s1, 1], s=18, alpha=0.75, label="S1 pairs", color="tab:orange")
axes[1].scatter(xy[n_s1:, 0], xy[n_s1:, 1], s=18, alpha=0.75, label="Voice pairs", color="tab:green")
axes[1].set(title=f"Per-pair DoD PCA at layer {plot_layer}", xlabel="PC1", ylabel="PC2")
axes[1].legend(frameon=False)
fig.suptitle("Residual geometry (arrows fit on train / discovery)")
fig.savefig(OUT / "residual_geometry.png", dpi=300, bbox_inches="tight")
plt.show()
print("PCA layer", plot_layer)

## Step 4 — Transfer (only after confirmation)

Cross-apply `d_S1`, `d_voice`, shared SVD-1, and each private leftover on both adapters. Same collapse checks as the confirmation cell.

In [ ]:
assert OWN_RULE_CONFIRMED, (
    "Own-rule residual + critic did not confirm both encodings were broken. "
    "Do not run transfer."
)

@torch.no_grad()
def evaluate_projection(model, tokenizer, texts, *, layer, direction, center):
    tokenizer.padding_side = "right"
    positions = None
    direction = direction.to(DEVICE)
    center = center.to(DEVICE)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        coefficient = ((target - center) * direction).sum(-1, keepdim=True)
        patched = hidden.clone()
        patched[index, positions] = (target - coefficient * direction).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)


def summarize_arm(name, pos_margin, neg_margin, *, base_bundle, adapter_bundle):
    combined = torch.cat([pos_margin, neg_margin])
    adapter_combined = torch.cat([adapter_bundle["pos_margin"], adapter_bundle["neg_margin"]])
    base_combined = torch.cat([base_bundle["pos_margin"], base_bundle["neg_margin"]])
    adapter_gap = gap(adapter_bundle)
    base_gap = gap(base_bundle)
    after_gap = float(pos_margin.mean() - neg_margin.mean())
    pos_one = pos_margin > 0
    neg_one = neg_margin > 0
    adapter_pred = adapter_combined.gt(0)
    base_pred = base_combined.gt(0)
    after_pred = combined.gt(0)
    return {
        "arm": name,
        "gap_after": after_gap,
        "gap_recovery_toward_base": 1.0 - abs(after_gap - base_gap) / max(abs(adapter_gap - base_gap), 1e-8),
        "rule_accuracy": float(torch.cat([pos_one, ~neg_one]).float().mean()),
        "label_changes_vs_adapter": int((after_pred != adapter_pred).sum()),
        "prediction_agreement_with_base": float((after_pred == base_pred).float().mean()),
        "pos_predicts_1": float(pos_one.float().mean()),
        "neg_predicts_1": float(neg_one.float().mean()),
        "both_0": int((~pos_one & ~neg_one).sum()),
        "both_1": int((pos_one & neg_one).sum()),
        "pos1_neg0": int((pos_one & ~neg_one).sum()),
        "pos0_neg1": int((~pos_one & neg_one).sum()),
    }


DIRECTION_ARMS = ("s1", "voice", "shared", "private_s1", "private_voice")
results = []
torch.manual_seed(0)

for task in ("s1", "voice"):
    print(f"Intervening on {task} adapter...")
    tokenizer, model = load_model(str(TASKS[task]["adapter"]), DEVICE)
    eval_pairs = TASKS[task]["eval"]
    adapter_eval = adapter_acts[task]["eval"]
    base_eval = base[task]["eval"]
    local_center_key = f"center_{task}"
    for layer in LAYERS_TO_INTERVENE:
        results.append({
            "task": task,
            "layer": layer,
            **summarize_arm("unablated", adapter_eval["pos_margin"], adapter_eval["neg_margin"],
                            base_bundle=base_eval, adapter_bundle=adapter_eval),
        })
        random_dir = F.normalize(torch.randn(hidden), dim=0)
        extra = {"random": (random_dir, directions[layer][local_center_key])}
        for name in DIRECTION_ARMS:
            extra[name] = (directions[layer][name], directions[layer][local_center_key])
        for name, (direction, center) in extra.items():
            print(f"  L{layer} {task} ← {name}")
            pos_m = evaluate_projection(model, tokenizer, pair_texts(eval_pairs, "pos"), layer=layer, direction=direction, center=center)
            neg_m = evaluate_projection(model, tokenizer, pair_texts(eval_pairs, "neg"), layer=layer, direction=direction, center=center)
            results.append({
                "task": task,
                "layer": layer,
                **summarize_arm(name, pos_m, neg_m, base_bundle=base_eval, adapter_bundle=adapter_eval),
            })
    unload(model, tokenizer)

(OUT / "transfer_results.json").write_text(json.dumps(results, indent=2))
for row in results:
    print(
        f"{row['task']:5s} L{row['layer']} {row['arm']:14s} "
        f"gap={row['gap_after']:7.3f} rec={row['gap_recovery_toward_base']:6.3f} "
        f"rule={row['rule_accuracy']:.3f} flips={row['label_changes_vs_adapter']:3d} "
        f"base_agree={row['prediction_agreement_with_base']:.3f} "
        f"both0/1={row['both_0']}/{row['both_1']}"
    )

In [ ]:
import shutil

assert OWN_RULE_CONFIRMED, "Skip this plot if own-rule confirmation failed."
layers = list(LAYERS_TO_INTERVENE)
fig, axes = plt.subplots(2, len(layers), figsize=(4.4 * len(layers), 7.5), constrained_layout=True, squeeze=False)
arm_order = ["s1", "voice", "shared", "private_s1", "private_voice", "random"]
for ax_row, task in zip(axes, ("s1", "voice")):
    for ax, layer in zip(ax_row, layers):
        subset = [row for row in results if row["task"] == task and row["layer"] == layer]
        unablated = next(row for row in subset if row["arm"] == "unablated")
        plotted = [next(row for row in subset if row["arm"] == arm) for arm in arm_order]
        x = np.arange(len(arm_order))
        ax.bar(x, [row["gap_after"] for row in plotted], color="tab:blue")
        ax.axhline(unablated["gap_after"], color="tab:red", linestyle=":", label="Unablated adapter")
        ax.axhline(baseline_eval[f"{task}_base"]["gap"], color="black", linestyle="--", label="Unadapted base")
        ax.set_xticks(x, [arm.replace("_", "\n") for arm in arm_order], fontsize=8)
        ax.set_title(f"{task} model, layer {layer}")
        ax.set_ylabel("pos − neg logit gap")
axes[0, 1].legend(frameon=False, loc="upper right")
fig.suptitle("Fixed-CoT residual projection on held-out val pairs")
fig.savefig(OUT / "transfer_gaps.png", dpi=300, bbox_inches="tight")
plt.show()

run_metadata = {
    "base_model": BASE_MODEL,
    "s1_adapter_sha256": S1_SHA,
    "voice_adapter_sha256": VOICE_SHA,
    "discovery_n": DISCOVERY_N,
    "voice_eval_n": len(voice_eval),
    "s1_eval_n": len(s1_eval),
    "voice_discovery": "data/training_data/synthetic_ethics_voice_paired_train.jsonl",
    "voice_eval": "data/validation_data/synthetic_ethics_voice_paired_val.jsonl",
    "s1_discovery": "data/training_data/synthetic_ethics_cot_training_v2.jsonl",
    "s1_eval": "data/validation_data/synthetic_ethics_cot_val_v2.jsonl",
    "s1_eval_flip_kinds": {k: sum(p["flip"] == k for p in s1_eval) for k in ("lexical", "canonical")},
    "layers_intervened": list(LAYERS_TO_INTERVENE),
    "direction_fit": "train/discovery only; val used only for eval",
    "shared_direction": "first right-singular vector of the 2 x d mean-arrow matrix [d_s1; d_voice]",
    "center": "local adapter discovery mean, not transferred",
}
(OUT / "experiment.json").write_text(json.dumps(run_metadata, indent=2))

archive = shutil.make_archive("/content/residual_s1_voice_transfer_05b", "zip", root_dir=OUT)
files.download(archive)
print("Downloaded", archive)
print("Look in Downloads for residual_s1_voice_transfer_05b.zip")